## Setup and Imports

**UPDATED: Added corrected `run()` and `_run()` methods with proper covariance handling per Smith & Zaldarriaga (2011)**

In [ ]:
import os
import sys
import numpy as np
import healpy as hp
import matplotlib.pyplot as plt
import h5py
from scipy.stats import norm
from scipy.interpolate import interp1d
from joblib import Parallel, delayed
from tqdm.auto import trange, tqdm

import camb
import lenspyx

from mlpng.core import Core
from mlpng.utils import (
    get_data,
    remove_mono_dipole,
    print_errors,
    save_data,
    make_alm_plots,
    plot_histogram,
    plot_predictions,
)
from ksw import KSW, Cosmology, ReducedBispectrum, Shape
from ksw.radial_functional import radial_func

plt.style.use("seaborn-v0_8-paper")

In [ ]:
class Generator(Core):
    """Generate Gaussian and non-Gaussian CMB alms with KSW estimator."""

    def __init__(self, argv=None, verbose=False):
        super().__init__(argv)

        # KSW batch size formula
        self.theta_batch = int(np.floor(1.5 * self.lmax + 1)) // self.slurm.n_cpus
        pols = self.pol_idxs()

        # Initialize cosmology
        camb_params = camb.set_params(**self.cosmo_params, verbose=verbose)
        self.cosmo: Cosmology = Cosmology(camb_params, verbose=verbose)

        # Compute transfer functions
        camb_lmax = max(self.lmax + self.lmax_buffer, 300)
        self.cosmo.compute_transfer(camb_lmax, verbose=verbose)
        self.cosmo.compute_c_ell(lmax=camb_lmax)

        # Store unlensed power spectra (theoretical C_ell from CAMB)
        # This is the signal power spectrum used for generating Gaussian alms
        c_ell = self.cosmo.c_ell["unlensed_scalar"]["c_ell"][: self.nell]
        self.c_ell = c_ell.T.astype(self.r_dtype)

        ic_ell = np.zeros_like(self.c_ell[pols])
        ic_ell[:, self.lmin :] = 1 / self.c_ell[pols, self.lmin :]
        self.ic_ell = remove_mono_dipole(ic_ell)

        # Compute unlensed covariance: Cov = B^2 * C_ell + N_ell
        # This is the observed covariance including beam and noise
        cov = self.b_ell**2 * self.c_ell + self.n_ell
        self.cov = remove_mono_dipole(cov)

        # Compute unlensed inverse covariance for KSW estimator weighting
        self.icov = np.zeros_like(self.cov[pols])
        self.icov[:, self.lmin :] = 1 / self.cov[pols, self.lmin :]

        # Lensing setup (if enabled)
        if self.lensing:
            # Lensed power spectra from CAMB (theoretical lensed C_ell)
            c_ell_lens = self.cosmo.c_ell["lensed_scalar"]["c_ell"][: self.nell]
            self.c_ell_lens = c_ell_lens.T.astype(self.r_dtype)

            # Lensed covariance: Cov_lens = B^2 * C_ell_lens + N_ell
            # Use this when analyzing lensed data for optimal weighting
            cov_lens = self.b_ell**2 * self.c_ell_lens + self.n_ell
            self.cov_lens = remove_mono_dipole(cov_lens)

            # Lensed inverse covariance for KSW estimator
            self.icov_lens = np.zeros_like(self.cov_lens[pols])
            self.icov_lens[:, self.lmin :] = 1 / self.cov_lens[pols, self.lmin :]

            # Lensing potential power spectrum
            self.cl_phi = self.cosmo.c_ell["lenspotential"]["c_ell"]
            self.cl_phi = self.cl_phi[:, 0].astype(self.r_dtype)

    def icov_func(self, alm, icov=None, lensed=False):
        """Apply inverse covariance to alms for KSW estimator."""
        if icov is None:
            icov = self.icov_lens if lensed else self.icov

        ret = np.zeros_like(alm)
        for pol in range(ret.shape[0]):
            ret[pol] = hp.almxfl(alm[pol], icov[pol])
        return ret

    @staticmethod
    def generate_single_alm(lmax, c_ells):
        """Generate a single Gaussian alm realization."""
        sims = hp.synalm(c_ells, lmax=lmax, new=True)
        sims = remove_mono_dipole(sims, True)
        return np.ascontiguousarray(sims)

    def generate_alm(self, nsims=None, c_ells=None) -> np.ndarray:
        """Generate Gaussian alms."""
        if nsims is None:
            nsims = self.nsims
        if c_ells is None:
            c_ells = self.cov

        sims = [hp.synalm(c_ells, lmax=self.lmax, new=True) for _ in range(nsims)]
        sims = remove_mono_dipole(np.array(sims))
        return np.ascontiguousarray(sims)

    def get_ksw(self, shape_str, train=False, n_steps=None, c_ells=None, lensed=False):
        """Create and initialize KSW estimator for given shape.

        Parameters:
            shape_str: Bispectrum shape name ("local", "equilateral", "orthogonal")
            train: If True, train the estimator using step_batch. If False, load from mc_file.
            n_steps: Number of MC steps for training. If None, uses self.mc_steps.
            c_ells: Covariance matrix for training (determines icov).
            lensed: If True, use lensed inverse covariance. If False, use unlensed.
        """
        ns = self.cosmo_params["ns"]
        ps = self.cosmo_params["pivot_scalar"]
        l_str = "lensed" if lensed else "unlensed"

        mc_dir = os.path.join(self.dirs["mc"], self.name)
        mc_file = os.path.join(mc_dir, f"{shape_str}-{l_str}")
        os.makedirs(mc_dir, exist_ok=True)

        match shape_str:
            case "local":
                shape = Shape.prim_local(ns, pivot=ps)
            case "equilateral":
                shape = Shape.prim_equilateral(ns, pivot=ps)
            case "orthogonal":
                shape = Shape.prim_orthogonal(ns, pivot=ps)
            case _:
                raise ValueError(f"Unknown shape {shape_str}")

        self.cosmo.red_bispectra = []
        self.cosmo.add_prim_reduced_bispectrum(shape, self.radii)

        ksw = KSW(
            self.cosmo.red_bispectra,
            lambda a: self.icov_func(a, None, lensed),
            self.lmax,
            self.pols,
            self.precision,
        )

        if train:
            nsteps = n_steps if n_steps is not None else self.mc_steps
            print(
                f"Training KSW estimator for {shape_str} (lensed={lensed}) with {nsteps} steps..."
            )

            if c_ells is None:
                c_ells = self.cov_lens if lensed else self.cov

            ksw.step_batch(
                lambda _: self.icov_func(
                    self.generate_single_alm(self.lmax, c_ells)[self.pol_idxs()],
                    None,
                    lensed,
                ),
                range(nsteps),
                comm=None,
                verbose=True,
                theta_batch=self.theta_batch,
            )

            # Save the trained state
            print(f"Saving trained KSW state to {mc_file}")
            ksw.write_state(mc_file)
        else:
            # Load from saved state
            ksw.start_from_read_state(mc_file)

        return ksw

    def train_all_shapes(self, n_steps=None, force=False):
        """Train KSW estimators for all shapes and both lensed/unlensed if lensing enabled.

        Parameters:
            n_steps: Number of MC steps for training. If None, uses self.mc_steps.
            force: If True, retrain even if mc_file exists.

        Returns:
            dict: Dictionary mapping (shape, lensed) tuples to trained KSW estimators
        """
        # Check if mc_file exists
        if os.path.exists(self.mc_file) and not force:
            print(
                f"MC file {self.mc_file} already exists, loading trained estimators (use force=True to retrain)"
            )
            trained_ksw = {}
            for shape in self.shapes:
                # Load unlensed estimator
                trained_ksw[(shape, False)] = self.get_ksw(
                    shape, train=False, lensed=False
                )
                # Load lensed estimator if lensing is enabled
                if self.lensing:
                    trained_ksw[(shape, True)] = self.get_ksw(
                        shape, train=False, lensed=True
                    )
            return trained_ksw

        # Remove existing file if forcing retrain
        if os.path.exists(self.mc_file) and force:
            print(f"Removing existing MC file {self.mc_file} for retraining")
            os.remove(self.mc_file)

        # Train all shapes with both lensed and unlensed
        trained_ksw = {}

        # Train unlensed first
        print(f"\n{'='*70}")
        print(f"Training UNLENSED estimators for all shapes")
        print(f"{'='*70}")
        for shape in self.shapes:
            print(f"\nTraining {shape} (unlensed)...")
            ksw = self.get_ksw(
                shape,
                train=True,
                n_steps=n_steps,
                c_ells=self.cov,
                lensed=False,
            )
            trained_ksw[(shape, False)] = ksw

        # Train lensed if lensing is enabled
        if self.lensing:
            print(f"\n{'='*70}")
            print(f"Training LENSED estimators for all shapes")
            print(f"{'='*70}")
            for shape in self.shapes:
                print(f"\nTraining {shape} (lensed)...")
                ksw = self.get_ksw(
                    shape,
                    train=True,
                    n_steps=n_steps,
                    c_ells=self.cov_lens,
                    lensed=True,
                )
                trained_ksw[(shape, True)] = ksw

        print(f"\n✓ Finished training all shapes!")
        return trained_ksw

    def lens_alms(self, alm, verbose=False):
        """Apply gravitational lensing to alms using lenspyx library.

        Parameters:
            alm: Input unlensed alms with shape (nsims, npols, nalm)
            verbose: Print verbose output

        Returns:
            (alm_lensed, alm_phi): Tuple of lensed alms and lensing potential alms
        """
        lmax = self.lmax + self.lmax_buffer
        fl = np.sqrt(np.arange(lmax + 1) * np.arange(1, lmax + 2))

        # Generate lensing potential alms
        cl_phi = self.cl_phi * self.phi_scale
        alm_phi = [
            hp.synalm(cl_phi, new=True, verbose=verbose) for _ in range(self.nsims)
        ]

        geom_info = ("healpix", {"nside": self.nside})
        geom = lenspyx.get_geom(geom_info)

        # Prepare output shape (handle E polarization if needed)
        alm_shape = alm.shape
        lensed_shape = (alm_shape[0], 3 if self.use_e else 1, alm_shape[2])
        alm_lensed = np.zeros(lensed_shape, dtype=alm.dtype)

        # Apply lensing to each simulation
        for sim in trange(self.nsims, desc="Lensing alms"):
            dlm = hp.almxfl(alm_phi[sim], fl)

            # Lens temperature component if enabled
            if self.use_t:
                lenmap = lenspyx.alm2lenmap(
                    alm[sim, 0],
                    dlm,
                    geometry=geom_info,
                    nthreads=self.slurm.n_cpus,
                )
                lenmap = np.ascontiguousarray(lenmap)

                alm_lensed[sim, 0] = geom.map2alm(
                    lenmap,
                    self.lmax,
                    self.lmax,
                    nthreads=self.slurm.n_cpus,
                )

            # Lens E/B components if enabled (limited support)
            if self.use_e:
                lenmap = lenspyx.alm2lenmap_spin(
                    alm[sim, 1:],
                    dlm,
                    geometry=geom_info,
                    nthreads=self.slurm.n_cpus,
                )
                lenmap = np.ascontiguousarray(lenmap)

                alm_lensed[sim, 1:] = geom.map2alm_spin(
                    lenmap,
                    2,
                    self.lmax,
                    self.lmax,
                    nthreads=self.slurm.n_cpus,
                )

        return alm_lensed, alm_phi

    def compute_fisher_shapes(self, shapes, lensed=False):
        """Compute Fisher information matrix for given bispectrum shapes.

        Parameters:
            shapes: List of bispectrum shape names
            lensed: If True, use lensed inverse covariance for weighting

        Returns:
            Fisher matrix (or scalar if single shape)
        """

        # get a lot of standard parameters from the cosmology
        tr_ell_k = self.cosmo.transfer["tr_ell_k"]
        k = self.cosmo.transfer["k"]
        ells_sparse = self.cosmo.transfer["ells"]
        ps = self.cosmo_params["pivot_scalar"]
        ns = self.cosmo_params["ns"]
        As = self.cosmo.camb_params.InitPower.As

        red_bispectra = []
        amp_factor = 2 * (2 * np.pi**2 * As) ** 2 * (3 / 5)

        for s in shapes:
            match s:
                case "local":
                    shape = Shape.prim_local(ns, pivot=ps)
                case "equilateral":
                    shape = Shape.prim_equilateral(ns, pivot=ps)
                case "orthogonal":
                    shape = Shape.prim_orthogonal(ns, pivot=ps)
                case _:
                    raise ValueError(f"Unknown shape {shape}")

            # this is cosmology.add_prim_reduced_bispectrum
            f_k = shape.get_f_k(k)
            amps = np.asarray(shape.amps).copy()
            amps *= amp_factor

            red_bisp = radial_func(f_k, tr_ell_k, k, self.radii, ells_sparse)
            factors, rule, weights = self.cosmo._parse_prim_reduced_bispec(
                red_bisp, self.radii, shape.rule, amps
            )
            red_bispectra.append(
                ReducedBispectrum(factors, rule, weights, ells_sparse, shape.name)
            )

        icov = self.icov_lens if lensed else self.icov
        ksw = self.get_ksw(shapes[0], train=False, lensed=lensed)
        return ksw.compute_fisher_multi(icov, red_bispectra)

    def _run(self, alm_l, lensed, verbose=False):
        """Generate non-Gaussian alms and estimates for given Gaussian alms.

        This matches the generator.py command structure exactly.

        Parameters:
            alm_l: Input Gaussian alms (unlensed if lensed=False, lensed if lensed=True)
            lensed: If True, use lensed covariance for optimal weighting
            verbose: Print verbose output
        """
        # Use correct power spectra based on lensed flag
        c_ells = self.c_ell
        icov = self.icov

        l_str = "lensed" if lensed else "unlensed"
        pol_idxs = self.pol_idxs()

        if self.slurm.is_main:
            print(f"Computing fisher matrix for {l_str} shapes: {self.shapes}")

        # Compute Fisher matrices for all shapes
        fisher_mat = np.atleast_1d(
            np.array(self.compute_fisher_shapes(self.shapes, lensed))
        )
        if len(self.shapes) > 1:
            marg_likes = np.sqrt(np.diag(np.linalg.inv(fisher_mat)))
        else:
            marg_likes = np.sqrt(1 / fisher_mat)

        print("fisher:", fisher_mat)
        print("margs:", marg_likes)

        sdata = {
            "fisher_matrix": {l_str: fisher_mat.astype(self.r_dtype)},
            "marginal_likelihoods": {l_str: marg_likes.astype(self.r_dtype)},
        }
        save_data(self.file, sdata, verbose=verbose)

        # Print 1-sigma confidence levels using pre-computed marginal likelihoods
        if self.slurm.is_main:
            print(f"\n1-sigma confidence levels ({l_str}):")
            for i, shape in enumerate(self.shapes):
                print(marg_likes)
                sigma = marg_likes[i]
                print(f"  {str(shape):12s}: σ_fnl = {sigma:.4f}")

        # Generate non-Gaussian alms for each shape
        for shape in self.shapes:
            print(f"\nProcessing {l_str} {shape}...")

            # Get KSW estimator
            ksw = self.get_ksw(shape, train=False, lensed=lensed)

            # Compute Fisher for this shape
            fisher = ksw.compute_fisher()
            print(f"  Fisher: {fisher}, σ: {1 / np.sqrt(fisher)}")

            # Compute non-Gaussian alms
            print(f"  Computing {l_str} alm_nl for shape {shape}...")
            if shape == "local":
                # use hanson method for local shape
                alm_nl = self.generate_alm_nl(alm_l, icov, lensed)
            else:
                # for the shapes we just use the KSW method
                alm_nl = self.generate_alm_nl_shape(alm_l, shape, ksw, icov, lensed)

            # Save without wrapping Fisher in list (this was causing HDF5 extension error)
            sdata = {
                "alm_nl": {l_str: {shape: alm_nl.astype(self.c_dtype)}},
                "fisher": {l_str: {shape: np.atleast_1d(fisher).astype(self.r_dtype)}},
            }
            save_data(self.file, sdata, verbose=verbose)

            if verbose:
                print(f"  Saved alm_nl/{l_str}/{shape} and fisher/{l_str}/{shape}")

            # Compute estimates if needed
            if self.slurm.is_main:
                print(f"  Computing estimates for {l_str} {shape}...")

                # Generate random fnl values
                fnl = self.rng.uniform(self.fnl_min, self.fnl_max, (self.nsims, 1, 1))
                alm = alm_l + fnl * alm_nl

                # Compute estimates
                n_estimates = min(self.nsims, self.num_estimates)
                estimates, _, _, _ = ksw.compute_estimate_batch(
                    lambda idx: self.icov_func(alm[idx, pol_idxs], icov, lensed),
                    range(n_estimates),
                    fisher=fisher,
                    theta_batch=self.theta_batch,
                )
                estimates = estimates.T

                # Save estimates
                sdata = {
                    "estimates": {l_str: {shape: estimates.astype(self.r_dtype)}},
                }
                save_data(self.file, sdata, verbose=verbose)

                print_errors(fnl, estimates, fisher)

    def run(self, verbose=False):
        """Main entry point: generate Gaussian alms, lens them, and compute non-Gaussian alms.

        This matches the generator.py command structure exactly.
        """
        pol_idxs = self.pol_idxs()

        # Generate unlensed Gaussian alms
        print("Generating unlensed Gaussian alms...")
        alm_l = self.generate_alm()

        sdata = {"alm_l": {"unlensed": alm_l[:, pol_idxs].astype(self.c_dtype)}}
        save_data(self.file, sdata, verbose=verbose)
        print(f"✓ Saved alm_l/unlensed with shape {alm_l[:, pol_idxs].shape}")

        # Process unlensed case
        print("\n" + "=" * 70)
        print("UNLENSED ANALYSIS")
        print("=" * 70)
        self._run(alm_l, False, verbose=verbose)

        # Process lensed case if lensing is enabled
        if self.lensing:
            print("\n" + "=" * 70)
            print("LENSING ALMS")
            print("=" * 70)

            alm_lens, alm_phi = self.lens_alms(alm_l, verbose=verbose)
            print(f"✓ Lensing complete. alm_lens shape: {alm_lens.shape}")

            # Save lensed alms
            sdata = {
                "alm_l": {"lensed": alm_lens[:, pol_idxs].astype(self.c_dtype)},
                "alm_phi": np.array(alm_phi).astype(self.c_dtype),
            }
            save_data(self.file, sdata, verbose=verbose)
            print(f"✓ Saved alm_l/lensed and alm_phi")

            # Process lensed case
            print("\n" + "=" * 70)
            print("LENSED ANALYSIS")
            print("=" * 70)
            self._run(alm_lens, True, verbose=verbose)

        print("\n" + "=" * 70)
        print("✓ Data generation complete!")
        print("=" * 70)

    def generate_alm_nl_shape(self, alms, shape, ksw=None, icov=None, lensed=False):
        """Generate non-Gaussian alms for given shape using KSW MC method.

        Parameters:
            alms: Input Gaussian alms
            shape: Bispectrum shape name
            ksw: KSW estimator (if None, will create one)
            icov: Inverse covariance to use (if None, will select based on lensed)
            lensed: If True, use lensed covariance

        Returns:
            Non-Gaussian alms array
        """
        pols = self.pol_idxs()

        if icov is None:
            icov = self.icov_lens if lensed else self.icov

        if ksw is None:
            ksw = self.get_ksw(shape, train=False, lensed=lensed)

        alm_ng = np.zeros_like(alms[:, pols])
        for i in trange(self.nsims, desc=f"alm_ng {shape}"):
            alm_icov = self.icov_func(alms[i, pols], icov=icov)
            alm_ng[i] = ksw.compute_ng_sim(alm_icov, theta_batch=self.theta_batch)

        return alm_ng

    def generate_alm_nl(self, alms, icov=None, lensed=False):
        """Generate non-Gaussian alms using Hanson 2010 radial integration method.

        This implements equation 27 from Hanson et al. 2010 for the local bispectrum shape.

        Parameters:
            alms: Input Gaussian alms
            icov: Inverse covariance (if None, will select based on lensed)
            lensed: If True, use lensed covariance for weighting

        Returns:
            Non-Gaussian alms array
        """
        # if icov is None:
        #     icov = self.icov_lens if lensed else self.icov

        pol_idxs = self.pol_idxs()
        transfer = self.cosmo.transfer

        # get the transfer functions with tr_ell_k being \Delta_\ell(k)
        tr_ells = transfer["ells"]
        tr_k = transfer["k"]
        tr_ell_k = transfer["tr_ell_k"]  # this is T, E, PHI
        tr_ell_k = tr_ell_k[..., pol_idxs]

        # camb returns transfers in zeta, need to convert to phi
        tr_ell_k *= 5 / 3

        # this will be the f(k) value to be placed in the radial function
        # first will be for alpha_ell, second will be beta_ell
        # see komatsu 2003 eq 5 and 6
        f_k = np.ones((len(tr_k), 2), dtype=self.r_dtype)

        # we need to multiply the f_k by the delta_phi term, and convert from k^2 to k^-1
        # see creminelii 2006, eq 16
        # for delta_phi = A: (taken from adri's ksw)
        # Planck defines A as <phi_k1 phi_k2> = (2pi)^3 delta(k12) A / k^3.
        # CAMB defines As as <zeta_k2 zeta_k2> = (2pi)^3 delta(k12) 2 * pi^2 As / k^3.
        # So A = (3/5)^2 * 2 * pi^2 As.
        Pk = self.cosmo.camb_params.primordial_power(tr_k, 0)
        delta_phi = 2 * np.pi**2 * Pk * (3 / 5) ** 2
        f_k[:, 1] = delta_phi * tr_k ** (-3)
        rad = radial_func(f_k, tr_ell_k, tr_k, self.radii, tr_ells)

        alpha_ell = rad[..., 0]
        alpha_l = interp1d(
            tr_ells,
            alpha_ell,
            "cubic",
            1,
            bounds_error=False,
            fill_value=0,
        )(self.ells)

        beta_ell = rad[..., 1]
        beta_l = interp1d(
            tr_ells,
            beta_ell,
            "cubic",
            1,
            bounds_error=False,
            fill_value=0,
        )(self.ells)

        bl_div_cl = np.zeros_like(beta_l)
        bl_div_cl[:, self.lmin :] = (
            beta_l[:, self.lmin :] * self.ic_ell.T[None, self.lmin :, ...]
        )

        # ensure all arrays are c contiguous, they wont be since we are using the interpolator which returns f contiguous
        alpha_l = np.ascontiguousarray(alpha_l)
        bl_div_cl = np.ascontiguousarray(bl_div_cl)

        # This uses joblib.parallel to generate the patches in parallel
        # by default (temp_folder=None) this will use a ram disk /dev/shm
        # if the data files are larger than the available memory, it will error
        # so we give it a temp folder to use, which wont have that problem
        temp_folder = os.environ.get("SCRATCH", None)
        parallel = Parallel(
            self.slurm.n_cpus,
            return_as="generator",
            temp_folder=temp_folder,
        )
        alm_nl = np.zeros_like(alms[:, pol_idxs])

        for sim in trange(self.nsims, desc="alm_nl"):
            gen = parallel(
                delayed(integrand)(
                    alms[sim, pol_idxs],
                    alpha_l[r],
                    bl_div_cl[r],
                    self.lmax,
                    self.nside,
                )
                for r in range(len(self.radii))
            )

            alm_nl[sim] = trap_generator(gen, self.radii)

        return alm_nl

## Configuration

In [ ]:
# ============================================================================
# Helper Functions (from generator.py)
# ============================================================================


def integrand(alm, alpha_l, bl_div_cl, lmax, nside, upscale=False):
    """Calculates the Outer integral of Hansen 2010 eq 27.

    That is: int dr r^2 [ alpha_ell(r) ( int d^2 hat{n} Y^*_{ell m}(hat{n}) B(r, hat{n})^2 ) ]

    Parameters:
        alm (ndarray): The input spherical harmonic coefficients.
        alpha_l (ndarray): The alpha_l coefficients.
        bl_div_cl (ndarray): The division of beta_ell function and C_l.
        lmax (int): The maximum l value.
        nside (int): The nside value.
        upscale (bool, optional): If True, the nside will be increased by 1 increment to improve calculations.

    Returns:
        ndarray: The calculated integrand.
    """
    if upscale:
        if nside >= 4096:
            raise ValueError(f"Nside {nside} is too large for upscaling")

        # valid nsides for use_pixel_weights
        nsides = np.array([32, 64, 128, 256, 512, 1024, 2048, 4096])
        nside = nsides[nsides > nside][0]

    Balm = np.array([hp.almxfl(alm[p], bl_div_cl[..., p]) for p in range(alm.shape[0])])
    B = hp.alm2map(Balm, nside, pol=False)
    inner = np.array(hp.map2alm(B**2, lmax, use_pixel_weights=True, pol=False), ndmin=2)
    solution = [hp.almxfl(inner[p], alpha_l[..., p]) for p in range(alpha_l.shape[-1])]
    return np.array(solution)


def trap_generator(gen, radii):
    """Perform trapezoidal integration using a generator.

    This will consume the memory as possible to help with memory management,
    this becomes needed for nside >= 512.

    Parameters:
        gen: A generator yielding the function values to integrate.
        radii: The radii values, should match the size of the generator

    Returns:
        The integral computed using trapezoidal rule.
    """
    integral = 0.0
    y_prev = radii[0] ** 2 * next(gen)
    for i in range(1, len(radii)):
        y_curr = radii[i] ** 2 * next(gen)
        integral += (radii[i] - radii[i - 1]) * (y_prev + y_curr) / 2.0
        y_prev = y_curr

    return integral

In [ ]:
# ============================================================================
# CONFIGURATION - Adjust these settings
# ============================================================================
USE_LENSING = True
NSIMS = 1
SHAPES = ["local", "equilateral", "orthogonal"]  # All shapes
# SHAPES = [""]
FORCE_GENERATION = True  # Set to True to regenerate data

# Initialize Generator with n64 settings
gen = Generator(
    [
        "./settings/n64.json",
        "--shapes",
        "all",  # Enable all shapes
        # "local",
        # "orthogonal",
        "--no-lensing",  # Enable lensing
        "--nsims",
        str(NSIMS),
        # "--no-estimate",  # We'll compute estimates manually in this notebook
    ],
    verbose=False,
)

print(f"Generator Configuration:")
print(f"  USE_LENSING: {USE_LENSING}")
print(f"  NSIMS: {NSIMS}")
print(f"  SHAPES: {SHAPES}")
print(f"  nside: {gen.nside}")
print(f"  lmax: {gen.lmax}")
print(f"  Data file: {gen.file}")

## Train KSW Estimator

In [ ]:
# ============================================================================
# KSW ESTIMATOR TRAINING
# ============================================================================
# Train KSW estimators for all shapes, both lensed and unlensed (if lensing enabled)
# Set FORCE_TRAIN=True to retrain even if mc_file exists

FORCE_TRAIN = True  # Set to True to force retraining
MC_STEPS = 1 #None  # Use default from settings, or set explicit number (e.g., 1000)

# Check if training is needed
if not os.path.exists(gen.mc_file) or FORCE_TRAIN:
    print(f"Training KSW estimators:")
    print(f"  Shapes: {gen.shapes}")
    print(f"  MC file: {gen.mc_file}")
    print(f"  MC steps: {MC_STEPS if MC_STEPS else gen.mc_steps} (default)")
    print(f"  Lensing enabled: {gen.lensing}")
    print(f"  Will train: unlensed + {'lensed' if gen.lensing else 'N/A'}")
    print()

    # Train all shapes with both lensed and unlensed variants
    trained_estimators = gen.train_all_shapes(n_steps=MC_STEPS, force=FORCE_TRAIN)

    print(f"\n✓ Training complete! Estimators saved to: {gen.mc_file}")
    print(f"Trained estimators: {list(trained_estimators.keys())}")
else:
    print(f"✓ MC file already exists: {gen.mc_file}")
    print(f"  Set FORCE_TRAIN=True to retrain")
    print(f"  Estimators will be loaded when needed")
    trained_estimators = {}

## Generate Data

In [ ]:
# Check if data already exists and delete to ensure clean state
# (prevents "TypeError: Scalar datasets cannot be extended" HDF5 error)
if os.path.exists(gen.file):
    print(f"Deleting existing data file to ensure clean state: {gen.file}")
    os.remove(gen.file)

# Always regenerate (FORCE_GENERATION controls whether to skip if file exists)
if FORCE_GENERATION or not os.path.exists(gen.file):
    print(f"Generating data...")
    gen.run(verbose=True)
else:
    print(f"✓ Data file exists: {gen.file}")
    with h5py.File(gen.file, "r") as f:
        print(f"  Keys: {list(f.keys())}")

## Load and Inspect Data

In [ ]:
# Load the generated data
l_str = "lensed" if USE_LENSING else "unlensed"

with h5py.File(gen.file, "r") as f:
    print(f"All available keys:")

    def print_keys(name, obj):
        print(f"  {name}")

    f.visititems(print_keys)

    print(f"\nData file structure ({l_str}):")

    # Check alm_l
    if f"alm_l/{l_str}" in f:
        alm_l_shape = f[f"alm_l/{l_str}"].shape
        print(f"  alm_l/{l_str}: {alm_l_shape}")

    # Check alm_nl and fisher for each shape
    for shape in SHAPES:
        if f"alm_nl/{l_str}/{shape}" in f:
            alm_nl_shape = f[f"alm_nl/{l_str}/{shape}"].shape
            print(f"  alm_nl/{l_str}/{shape}: {alm_nl_shape}")

        if f"fisher/{l_str}/{shape}" in f:
            fisher_val = f[f"fisher/{l_str}/{shape}"][()]
            print(f"  fisher/{l_str}/{shape}: {fisher_val}")

## Compute KSW Estimates

For each shape, compute fnl estimates using KSW estimator.

In [ ]:
# Load data and compute estimates for each shape and lensing variant
# Train with matching icov: unlensed data uses unlensed estimator, lensed data uses lensed estimator
results = {}

# Process unlensed data
print(f"\n{'='*70}")
print(f"UNLENSED ANALYSIS")
print(f"{'='*70}")
for shape in SHAPES:
    print(f"\n{'-'*70}")
    print(f"Processing {shape.upper()} shape (unlensed)")
    print(f"{'-'*70}")

    # Load unlensed alms
    alm_l = get_data(gen.file, f"alm_l/unlensed", np.arange(0, NSIMS))
    alm_nl = get_data(gen.file, f"alm_nl/unlensed/{shape}", np.arange(0, NSIMS))

    with h5py.File(gen.file, "r") as f:
        fisher = f[f"fisher/unlensed/{shape}"][()]

    print(f"Loaded data:")
    print(f"  alm_l shape: {alm_l.shape}")
    print(f"  alm_nl shape: {alm_nl.shape}")
    print(f"  fisher: {fisher}")

    # Generate random fnl values
    fnl = gen.rng.uniform(gen.fnl_min, gen.fnl_max, (NSIMS, 1, 1))

    # Combine alms: alm = alm_l + fnl * alm_nl
    alm = alm_l + fnl * alm_nl

    # Get KSW estimator trained with unlensed icov
    print(f"Loading KSW estimator trained with unlensed icov...")
    ksw = gen.get_ksw(shape, train=False, lensed=False)

    # Compute estimates
    print(f"Computing {NSIMS} KSW estimates...")
    theta_batch = int(np.floor(1.5 * gen.lmax + 1))

    estimates, _, _, _ = ksw.compute_estimate_batch(
        lambda idx: gen.icov_func(alm[idx], lensed=False),
        range(NSIMS),
        comm=None,
        fisher=fisher,
        theta_batch=theta_batch,
    )
    estimates = estimates.T

    print(f"  estimates shape: {estimates.shape}")
    print(f"  estimates mean: {estimates.mean():.2f}")
    print(f"  estimates std: {estimates.std():.2f}")

    # Store results
    fisher_scalar = np.asarray(fisher).item()
    sigma_scalar = float(1.0 / np.sqrt(fisher_scalar))
    results[(shape, "unlensed")] = {
        "fnl": fnl.flatten(),
        "estimates": estimates,
        "fisher": fisher_scalar,
        "sigma": sigma_scalar,
    }

    # Print errors
    print(f"\nEstimation Results:")
    print_errors(fnl, estimates, fisher)

# Process lensed data if lensing is enabled
if gen.lensing:
    print(f"\n{'='*70}")
    print(f"LENSED ANALYSIS")
    print(f"{'='*70}")
    for shape in SHAPES:
        print(f"\n{'-'*70}")
        print(f"Processing {shape.upper()} shape (lensed)")
        print(f"{'-'*70}")

        # Load lensed alms
        alm_l = get_data(gen.file, f"alm_l/lensed", np.arange(0, NSIMS))
        alm_nl = get_data(gen.file, f"alm_nl/lensed/{shape}", np.arange(0, NSIMS))

        with h5py.File(gen.file, "r") as f:
            fisher = f[f"fisher/lensed/{shape}"][()]

        print(f"Loaded data:")
        print(f"  alm_l shape: {alm_l.shape}")
        print(f"  alm_nl shape: {alm_nl.shape}")
        print(f"  fisher: {fisher}")

        # Generate random fnl values
        fnl = gen.rng.uniform(gen.fnl_min, gen.fnl_max, (NSIMS, 1, 1))

        # Combine alms: alm = alm_l + fnl * alm_nl
        alm = alm_l + fnl * alm_nl

        # Get KSW estimator trained with lensed icov
        print(f"Loading KSW estimator trained with lensed icov...")
        ksw = gen.get_ksw(shape, train=False, lensed=True)

        # Compute estimates
        print(f"Computing {NSIMS} KSW estimates...")
        theta_batch = int(np.floor(1.5 * gen.lmax + 1))

        estimates, _, _, _ = ksw.compute_estimate_batch(
            lambda idx: gen.icov_func(alm[idx], lensed=True),
            range(NSIMS),
            comm=None,
            fisher=fisher,
            theta_batch=theta_batch,
        )
        estimates = estimates.T

        print(f"  estimates shape: {estimates.shape}")
        print(f"  estimates mean: {estimates.mean():.2f}")
        print(f"  estimates std: {estimates.std():.2f}")

        # Store results
        fisher_scalar = np.asarray(fisher).item()
        sigma_scalar = float(1.0 / np.sqrt(fisher_scalar))
        results[(shape, "lensed")] = {
            "fnl": fnl.flatten(),
            "estimates": estimates,
            "fisher": fisher_scalar,
            "sigma": sigma_scalar,
        }

        # Print errors
        print(f"\nEstimation Results:")
        print_errors(fnl, estimates, fisher)

## Visualization: Predictions with Uncertainty Bands

In [ ]:
# Create prediction plots for each shape and lensing variant
data_types = ["unlensed"]
if gen.lensing:
    data_types.append("lensed")

fig, axes = plt.subplots(
    len(SHAPES), len(data_types), figsize=(5 * len(data_types), 5 * len(SHAPES))
)
if len(SHAPES) == 1 and len(data_types) == 1:
    axes = np.array([[axes]])
elif len(SHAPES) == 1:
    axes = axes.reshape(1, -1)
elif len(data_types) == 1:
    axes = axes.reshape(-1, 1)

for shape_idx, shape in enumerate(SHAPES):
    for dtype_idx, dtype in enumerate(data_types):
        # Ensure axes is always 2D for consistent indexing
        ax = axes[shape_idx, dtype_idx]

        key = (shape, dtype)
        if key not in results:
            continue

        fnl = results[key]["fnl"]
        estimates = results[key]["estimates"]
        sigma = results[key]["sigma"]

        # Sort by true fnl for plotting
        sort_idx = np.argsort(fnl)
        fnl_sorted = fnl[sort_idx]
        est_sorted = estimates[sort_idx]

        # Plot points
        ax.scatter(fnl_sorted, est_sorted, alpha=0.6, s=20, label="Estimates")

        # Plot perfect reconstruction line
        fnl_range = np.array([fnl_sorted.min(), fnl_sorted.max()])
        ax.plot(
            fnl_range, fnl_range, "r--", linewidth=2, label="Perfect reconstruction"
        )

        # Add uncertainty band
        ax.fill_between(
            fnl_range,
            fnl_range - sigma,
            fnl_range + sigma,
            alpha=0.2,
            color="red",
            label=f"±σ (σ={sigma:.2f})",
        )

        ax.set_xlabel("True $f_{NL}$", fontsize=12)
        ax.set_ylabel("Estimated $f_{NL}$", fontsize=12)
        ax.set_title(f"{shape.capitalize()} ({dtype})", fontsize=14)
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("./data/plots/generator_test_predictions.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved: ./data/plots/generator_test_predictions.png")

## Visualization: Error Histograms

In [ ]:
# Create error histograms for each shape and lensing variant
data_types = ["unlensed"]
if gen.lensing:
    data_types.append("lensed")

fig, axes = plt.subplots(
    len(SHAPES), len(data_types), figsize=(5 * len(data_types), 5 * len(SHAPES))
)
if len(SHAPES) == 1 and len(data_types) == 1:
    axes = np.array([[axes]])
elif len(SHAPES) == 1:
    axes = axes.reshape(1, -1)
elif len(data_types) == 1:
    axes = axes.reshape(-1, 1)

for shape_idx, shape in enumerate(SHAPES):
    for dtype_idx, dtype in enumerate(data_types):
        ax = axes[shape_idx, dtype_idx]
        key = (shape, dtype)
        if key not in results:
            continue

        fnl = results[key]["fnl"]
        estimates = results[key]["estimates"]
        sigma = results[key]["sigma"]

        # Compute errors
        errors = (estimates - fnl) / sigma

        # Plot histogram
        ax.hist(
            errors, bins=20, alpha=0.7, density=True, edgecolor="black", label="Errors"
        )

        # Overlay standard normal
        x = np.linspace(errors.min(), errors.max(), 100)
        ax.plot(x, norm.pdf(x), "r-", linewidth=2, label="Standard Normal")

        ax.set_xlabel("$(f_{NL}^{est} - f_{NL}^{true}) / \sigma$", fontsize=12)
        ax.set_ylabel("Density", fontsize=12)
        ax.set_title(f"{shape.capitalize()} ({dtype})", fontsize=14)
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("./data/plots/generator_test_errors.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved: ./data/plots/generator_test_errors.png")

## Summary Statistics

In [ ]:
print("\n" + "=" * 70)
print("SUMMARY STATISTICS")
print("=" * 70 + "\n")

data_types = ["unlensed"]
if gen.lensing:
    data_types.append("lensed")

for shape in SHAPES:
    for dtype in data_types:
        key = (shape, dtype)
        if key not in results:
            continue

        fnl = results[key]["fnl"]
        estimates = results[key]["estimates"]
        sigma = results[key]["sigma"]
        fisher = results[key]["fisher"]

        # Compute metrics
        bias = np.mean(estimates - fnl)
        rms = np.sqrt(np.mean((estimates - fnl) ** 2))
        errors_normalized = (estimates - fnl) / sigma
        chi2 = np.mean(errors_normalized**2)

        # Compute percentage within 1-sigma and 2-sigma bands
        within_1sigma = np.sum(np.abs(estimates - fnl) <= sigma) / len(fnl) * 100
        within_2sigma = np.sum(np.abs(estimates - fnl) <= 2 * sigma) / len(fnl) * 100
        n_within_1sigma = int(np.sum(np.abs(estimates - fnl) <= sigma))
        n_within_2sigma = int(np.sum(np.abs(estimates - fnl) <= 2 * sigma))

        print(f"{shape.upper()} ({dtype}):")
        print(f"  Fisher: {fisher:.6e}")
        print(f"  σ (1-σ uncertainty): {sigma:.4f}")
        print(f"  Bias: {bias:.4f}")
        print(f"  RMS error: {rms:.4f}")
        print(f"  χ²/dof: {chi2:.4f}")
        print(f"  Within 1-σ: {within_1sigma:.1f}% ({n_within_1sigma}/{len(fnl)})")
        print(f"  Within 2-σ: {within_2sigma:.1f}% ({n_within_2sigma}/{len(fnl)})")
        print(f"  True fnl range: [{fnl.min():.1f}, {fnl.max():.1f}]")
        print(f"  Est fnl range: [{estimates.min():.1f}, {estimates.max():.1f}]")
        print()

## Comparison: Lensed vs Unlensed (if lensing enabled)

Compare the estimation performance between lensed and unlensed data.

In [ ]:
# Compare lensed vs unlensed results if both are available
if gen.lensing:
    print("\n" + "=" * 70)
    print("COMPARISON: LENSED vs UNLENSED")
    print("=" * 70 + "\n")

    for shape in SHAPES:
        print(f"\n{shape.upper()} Comparison:")

        # Unlensed metrics
        key_unl = (shape, "unlensed")
        fnl_u = results[key_unl]["fnl"]
        est_u = results[key_unl]["estimates"]
        sigma_u = results[key_unl]["sigma"]
        fisher_u = results[key_unl]["fisher"]
        bias_u = np.mean(est_u - fnl_u)
        rms_u = np.sqrt(np.mean((est_u - fnl_u) ** 2))
        chi2_u = np.mean(((est_u - fnl_u) / sigma_u) ** 2)

        # Lensed metrics
        key_l = (shape, "lensed")
        fnl_l = results[key_l]["fnl"]
        est_l = results[key_l]["estimates"]
        sigma_l = results[key_l]["sigma"]
        fisher_l = results[key_l]["fisher"]
        bias_l = np.mean(est_l - fnl_l)
        rms_l = np.sqrt(np.mean((est_l - fnl_l) ** 2))
        chi2_l = np.mean(((est_l - fnl_l) / sigma_l) ** 2)

        print(f"  {'Metric':<20} {'Unlensed':>12} {'Lensed':>12} {'Δ (L-U)':>12}")
        print(f"  {'-'*20} {'-'*12} {'-'*12} {'-'*12}")
        print(
            f"  {'Fisher':<20} {fisher_u:>12.4e} {fisher_l:>12.4e} {fisher_l-fisher_u:>12.4e}"
        )
        print(f"  {'σ':<20} {sigma_u:>12.4f} {sigma_l:>12.4f} {sigma_l-sigma_u:>12.4f}")
        print(f"  {'Bias':<20} {bias_u:>12.4f} {bias_l:>12.4f} {bias_l-bias_u:>12.4f}")
        print(f"  {'RMS error':<20} {rms_u:>12.4f} {rms_l:>12.4f} {rms_l-rms_u:>12.4f}")
        print(
            f"  {'χ²/dof':<20} {chi2_u:>12.4f} {chi2_l:>12.4f} {chi2_l-chi2_u:>12.4f}"
        )
else:
    print("Lensing disabled - no comparison available")

### Unlensed: Predictions with Uncertainty Bands

In [ ]:
# Create prediction plots for unlensed estimates
fig, axes = plt.subplots(1, len(SHAPES), figsize=(5 * len(SHAPES), 5))
if len(SHAPES) == 1:
    axes = [axes]

for idx, shape in enumerate(SHAPES):
    ax = axes[idx]
    fnl = results[(shape, "unlensed")]["fnl"]
    estimates = results[(shape, "unlensed")]["estimates"]
    sigma = results[(shape, "unlensed")]["sigma"]

    # Sort by true fnl for plotting
    sort_idx = np.argsort(fnl)
    fnl_sorted = fnl[sort_idx]
    est_sorted = estimates[sort_idx]

    # Plot points
    ax.scatter(fnl_sorted, est_sorted, alpha=0.6, s=20, label="Estimates")

    # Plot perfect reconstruction line
    fnl_range = np.array([fnl_sorted.min(), fnl_sorted.max()])
    ax.plot(fnl_range, fnl_range, "r--", linewidth=2, label="Perfect reconstruction")

    # Add uncertainty band
    ax.fill_between(
        fnl_range,
        fnl_range - sigma,
        fnl_range + sigma,
        alpha=0.2,
        color="red",
        label=f"±σ (σ={sigma:.2f})",
    )

    ax.set_xlabel("True $f_{NL}$", fontsize=12)
    ax.set_ylabel("Estimated $f_{NL}$", fontsize=12)
    ax.set_title(f"{shape.capitalize()} (unlensed)", fontsize=14)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(
    "./data/plots/generator_test_predictions_unlensed.png", dpi=150, bbox_inches="tight"
)
plt.show()
print("✓ Saved: ./data/plots/generator_test_predictions_unlensed.png")

### Unlensed: Error Histograms

In [ ]:
# Create error histograms for unlensed estimates
fig, axes = plt.subplots(1, len(SHAPES), figsize=(5 * len(SHAPES), 5))
if len(SHAPES) == 1:
    axes = [axes]

for idx, shape in enumerate(SHAPES):
    ax = axes[idx]
    fnl = results[(shape, "unlensed")]["fnl"]
    estimates = results[(shape, "unlensed")]["estimates"]
    sigma = results[(shape, "unlensed")]["sigma"]

    # Compute errors
    errors = (estimates - fnl) / sigma

    # Plot histogram
    ax.hist(errors, bins=20, alpha=0.7, density=True, edgecolor="black", label="Errors")

    # Overlay standard normal
    x = np.linspace(errors.min(), errors.max(), 100)
    ax.plot(x, norm.pdf(x), "r-", linewidth=2, label="Standard Normal")

    ax.set_xlabel("$(f_{NL}^{est} - f_{NL}^{true}) / \sigma$", fontsize=12)
    ax.set_ylabel("Density", fontsize=12)
    ax.set_title(f"{shape.capitalize()} (unlensed)", fontsize=14)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(
    "./data/plots/generator_test_errors_unlensed.png", dpi=150, bbox_inches="tight"
)
plt.show()
print("✓ Saved: ./data/plots/generator_test_errors_unlensed.png")

### Unlensed: Summary Statistics

In [ ]:
print("\n" + "=" * 70)
print("UNLENSED SUMMARY STATISTICS")
print("=" * 70 + "\n")

for shape in SHAPES:
    fnl = results[(shape, "unlensed")]["fnl"]
    estimates = results[(shape, "unlensed")]["estimates"]
    sigma = results[(shape, "unlensed")]["sigma"]
    fisher = results[(shape, "unlensed")]["fisher"]

    # Compute metrics
    bias = np.mean(estimates - fnl)
    rms = np.sqrt(np.mean((estimates - fnl) ** 2))
    errors_normalized = (estimates - fnl) / sigma
    chi2 = np.mean(errors_normalized**2)

    print(f"{shape.upper()}:")
    print(f"  Fisher: {fisher:.6e}")
    print(f"  σ (1-σ uncertainty): {sigma:.4f}")
    print(f"  Bias: {bias:.4f}")
    print(f"  RMS error: {rms:.4f}")
    print(f"  χ²/dof: {chi2:.4f}")
    print(f"  True fnl range: [{fnl.min():.1f}, {fnl.max():.1f}]")
    print(f"  Est fnl range: [{estimates.min():.1f}, {estimates.max():.1f}]")
    print()

## Comparison: Lensed vs Unlensed

Compare the estimation performance between lensed and unlensed data.

In [ ]:
# Compare lensed vs unlensed results
if USE_LENSING:
    print("\n" + "=" * 70)
    print("COMPARISON: LENSED vs UNLENSED")
    print("=" * 70 + "\n")

    for shape in SHAPES:
        print(f"{shape.upper()}:")

        # Lensed metrics
        fnl_l = results[(shape, "lensed")]["fnl"]
        est_l = results[(shape, "lensed")]["estimates"]
        sigma_l = results[(shape, "lensed")]["sigma"]
        fisher_l = results[(shape, "lensed")]["fisher"]
        bias_l = np.mean(est_l - fnl_l)
        rms_l = np.sqrt(np.mean((est_l - fnl_l) ** 2))
        chi2_l = np.mean(((est_l - fnl_l) / sigma_l) ** 2)

        # Unlensed metrics
        fnl_u = results[(shape, "unlensed")]["fnl"]
        est_u = results[(shape, "unlensed")]["estimates"]
        sigma_u = results[(shape, "unlensed")]["sigma"]
        fisher_u = results[(shape, "unlensed")]["fisher"]
        bias_u = np.mean(est_u - fnl_u)
        rms_u = np.sqrt(np.mean((est_u - fnl_u) ** 2))
        chi2_u = np.mean(((est_u - fnl_u) / sigma_u) ** 2)

        print(f"  {'Metric':<20} {'Lensed':>12} {'Unlensed':>12} {'Δ (L-U)':>12}")
        print(f"  {'-'*20} {'-'*12} {'-'*12} {'-'*12}")
        print(
            f"  {'Fisher':<20} {fisher_l:>12.4e} {fisher_u:>12.4e} {fisher_l-fisher_u:>12.4e}"
        )
        print(f"  {'σ':<20} {sigma_l:>12.4f} {sigma_u:>12.4f} {sigma_l-sigma_u:>12.4f}")
        print(f"  {'Bias':<20} {bias_l:>12.4f} {bias_u:>12.4f} {bias_l-bias_u:>12.4f}")
        print(f"  {'RMS error':<20} {rms_l:>12.4f} {rms_u:>12.4f} {rms_l-rms_u:>12.4f}")
        print(
            f"  {'χ²/dof':<20} {chi2_l:>12.4f} {chi2_u:>12.4f} {chi2_l-chi2_u:>12.4f}"
        )
        print()
else:
    print("Lensing disabled - no comparison available")